In [1]:
!pip install -q google-genai

In [2]:
from google import genai
from google.colab import userdata

api_key = userdata.get('GEMINI_API_KEY')
client = genai.Client(api_key=api_key)

In [ ]:
response = client.models.generate_content(
    model="gemini-3.5-flash",
    contents="안녕. 너를 간단히 소개해봐."
)
print(response.text)

안녕하세요, 저는 구글에서 개발한 인공지능 언어 모델입니다.


In [3]:
def make_prompt(review):
    prompt = f"""너는 무인 스터디카페 사장이야.
아래 손님 리뷰를 읽고 4가지를 판단해줘.

[가게 정보]
- 가게 이름: 카공족 건대로데오점
- 요금: 24시간 이용권 5,900원(온라인 결제만 가능, 실물카드 불가)
- 무료 제공: 커피, 차, 아이스티, 얼음, 쿠키, 정수기
- 시설: 매장 내부 화장실(남녀공용 1개), 개인 스탠드, 독서대, 충전기
- 없는 것: 사물함, 스터디룸
[판단 기준]
1. 카테고리: 청결, 분위기·소음, 좌석·공간, 냉난방, 시설·비품, 결제·요금, 기타 중 하나
2. 감정: 긍정 또는 부정
   - 불만이나 아쉬움이 조금이라도 섞여 있으면 부정
   - 시설이나 서비스가 있는지 묻거나 요청하는 질문도 부정
3. 긴급도: 높음, 보통, 낮음 중 하나
   - 높음: 고장, 결제 불가처럼 지금 바로 조치가 필요한 것
   - 보통: 불편 사항이나 개선 요청
   - 낮음: 칭찬이나 단순 의견
4. 답글: 사장 입장에서 쓴 3~4문장의 답글
   - 말투: 상냥하고 친절하되 과하지 않은 정중한 존댓말
   - 손님(유저)이름을 언급하지 말 것
   - 느낌표, 이모티콘을 문장 중간에 적절히 사용
   - 리뷰에서 손님이 언급한 내용을 구체적으로 짚어서 답할 것
   - 불만이 있는 리뷰에는 사과와 함께 확인하겠다는 내용을 넣을 것
   - 가게 정보에 없는 내용은 약속하지 말 것
   - 줄바꿈 없이 한 줄로 이어서 쓸 것

[출력 형식]
다른 말 없이 아래처럼 한 줄로만 답해줘.
카테고리|평가|긴급도|답글

[리뷰]
{review}"""
    return prompt

In [ ]:
review = "커피머신과 정수기가 작동하지 않습니다."

response = client.models.generate_content(
    model="gemini-3.5-flash",
    contents=make_prompt(review)
)
print(response.text)

시설·비품|부정|높음|저희 매장의 커피머신과 정수기가 작동하지 않아 큰 불편을 드려 고개 숙여 사과드립니다.😭 해당 장비는 즉시 현장 점검을 실시하여 빠르게 정상화하도록 조치하겠습니다! 이용에 차질을 드린 점 다시 한번 죄송하며, 쾌적한 환경을 위해 더욱 노력하는 카공족 건대로데오점이 되겠습니다.🙏


In [ ]:
result = response.text.strip().split("|")

print(result)
print("카테고리:", result[0])
print("평가:", result[1])
print("긴급도:", result[2])
print("답글:", result[3])

['시설·비품', '부정', '높음', '저희 매장의 커피머신과 정수기가 작동하지 않아 큰 불편을 드려 고개 숙여 사과드립니다.😭 해당 장비는 즉시 현장 점검을 실시하여 빠르게 정상화하도록 조치하겠습니다! 이용에 차질을 드린 점 다시 한번 죄송하며, 쾌적한 환경을 위해 더욱 노력하는 카공족 건대로데오점이 되겠습니다.🙏']
카테고리: 시설·비품
평가: 부정
긴급도: 높음
답글: 저희 매장의 커피머신과 정수기가 작동하지 않아 큰 불편을 드려 고개 숙여 사과드립니다.😭 해당 장비는 즉시 현장 점검을 실시하여 빠르게 정상화하도록 조치하겠습니다! 이용에 차질을 드린 점 다시 한번 죄송하며, 쾌적한 환경을 위해 더욱 노력하는 카공족 건대로데오점이 되겠습니다.🙏


In [4]:
import pandas as pd

df = pd.read_csv("reviews.csv", encoding="utf-8-sig")
print(len(df))
df.head()

90


,id,date,review,label,source
0,1,NaN,항상 잘 이용하고 있습니다. 앞으로도 자주 올게요,긍정,naver
1,2,NaN,화장실이 내부에 있어서 좋고 깔끔하게 관리가 잘 되고 있어요. 좋습니댱,긍정,naver
2,3,NaN,조용하고 깔끔해서 집중하기 좋아요,긍정,naver
3,4,NaN,매장이 깔끔하고 공부에 집중하기 너무 좋은 분위기였어요! 다음에 또 방문하고싶어요🤍,긍정,naver
4,5,NaN,생각했던 것보다 훨씬 좋아요!!!! 조용하고 모두 열심히 집중하고 계시니 저도 덩달...,긍정,naver


In [5]:
import time

def ask_gemini(review):
    for attempt in range(5):
        try:
            response = client.models.generate_content(
                model="gemini-3.5-flash-lite",
                contents=make_prompt(review)
            )
            return response.text.strip()
        except Exception as e:
            print(attempt + 1, "번째 시도 실패:", e)
            time.sleep(30)
    return "오류|오류|오류|오류"

In [ ]:
print(ask_gemini(df["review"][0]))

기타|긍정|낮음|저희 카공족 건대로데오점을 늘 애용해 주시고 이렇게 따뜻한 리뷰까지 남겨주셔서 정말 감사합니다! 😊 항상 잘 이용하고 계신다는 말씀에 사장으로서 큰 힘과 보람을 느낍니다. ✨ 앞으로도 자주 방문해 주실 때마다 늘 쾌적하고 공부하기 좋은 공간이 될 수 있도록 정성껏 관리하겠습니다. ☕ 오늘도 목표하신 공부 모두 이루시고 행복한 하루 보내세요! 🙏


In [6]:
def split_result(text):
    parts = text.split("|")
    if len(parts) == 4:
        return [parts[0].strip(), parts[1].strip(), parts[2].strip(), parts[3].strip()]
    else:
        return ["형식오류", "형식오류", "형식오류", text]

In [ ]:
for review in df["review"].head(3):
    text = ask_gemini(review)
    parts = split_result(text)

    print("리뷰:", review)
    print("→", parts[0], "/", parts[1], "/", parts[2])
    print("→ 답글:", parts[3])
    print("----------")

    time.sleep(5)

리뷰: 항상 잘 이용하고 있습니다. 앞으로도 자주 올게요
→ 기타 / 긍정 / 낮음
→ 답글: 안녕하세요, 카공족 건대로데오점을 항상 애용해 주셔서 정말 감사합니다! 보내주신 따뜻한 말씀 덕분에 더욱 힘내서 운영하게 되네요😊 앞으로도 편안하고 쾌적한 공부 공간이 될 수 있도록 항상 노력할 테니, 언제든 필요할 때 편하게 찾아주세요!
----------
리뷰: 화장실이 내부에 있어서 좋고 깔끔하게 관리가 잘 되고 있어요. 좋습니댱
→ 시설·비품 / 긍정 / 낮음
→ 답글: 카공족 건대로데오점을 찾아주셔서 정말 감사합니다! 매장 내부 화장실이 깔끔하게 관리되고 있어 마음에 드셨다니 다행이에요. 앞으로도 쾌적하고 편안한 공간을 유지하기 위해 항상 노력하겠습니다! 공부하시다가 언제든 또 방문해 주세요!
----------
리뷰: 조용하고 깔끔해서 집중하기 좋아요
→ 좌석·공간 / 긍정 / 낮음
→ 답글: 카공족 건대로데오점에서 조용하고 깔끔한 분위기 속에서 집중이 잘 되셨다니 정말 기쁘네요! 앞으로도 편안하게 공부하실 수 있도록 항상 쾌적한 환경을 유지하기 위해 노력하겠습니다. 다음에도 열공하러 또 찾아주세요~!
----------


In [7]:
categories = []
ai_labels = []
urgencies = []
replies = []

count = 0
for review in df["review"]:
    text = ask_gemini(review)
    parts = split_result(text)

    categories.append(parts[0])
    ai_labels.append(parts[1])
    urgencies.append(parts[2])
    replies.append(parts[3])

    count = count + 1
    print(count, "/", len(df), "완료:", parts[0], parts[1], parts[2])

    time.sleep(5)

1 / 90 완료: 기타 긍정 낮음
2 / 90 완료: 청결 긍정 낮음
3 / 90 완료: 좌석·공간 긍정 낮음
4 / 90 완료: 청결 긍정 낮음
5 / 90 완료: 분위기·소음 긍정 낮음
6 / 90 완료: 좌석·공간 긍정 낮음
7 / 90 완료: 분위기·소음 긍정 낮음
8 / 90 완료: 청결 긍정 낮음
9 / 90 완료: 결제·요금 긍정 낮음
10 / 90 완료: 좌석·공간 긍정 낮음
11 / 90 완료: 좌석·공간 부정 낮음
12 / 90 완료: 카테고리: 분위기·소음 감정: 긍정 긴급도: 낮음
13 / 90 완료: 좌석·공간 긍정 낮음
14 / 90 완료: 좌석·공간 긍정 낮음
15 / 90 완료: 좌석·공간 긍정 낮음
16 / 90 완료: 기분·소음 긍정 낮음
17 / 90 완료: 좌석·공간 긍정 낮음
18 / 90 완료: 결제·요금 긍정 낮음
19 / 90 완료: 분위기·소음 긍정 낮음
20 / 90 완료: 분위기·소음 긍정 낮음
21 / 90 완료: 좌석·공간 긍정 낮음
22 / 90 완료: 청결 긍정 낮음
23 / 90 완료: 분위기·소음 긍정 낮음
24 / 90 완료: 분위기·소음 긍정 낮음
25 / 90 완료: 좌석·공간 부정 보통
26 / 90 완료: 좌석·공간 긍정 낮음
27 / 90 완료: 시설·비품 긍정 낮음
28 / 90 완료: 좌석·공간 긍정 낮음
29 / 90 완료: 분위기·소음 긍정 낮음
30 / 90 완료: 좌석·공간 긍정 낮음
31 / 90 완료: 좌석·공간 긍정 낮음
32 / 90 완료: 좌석·공간 부정 보통
33 / 90 완료: 분위기·소음 긍정 낮음
34 / 90 완료: 분위기·소음 긍정 낮음
35 / 90 완료: 좌석·공간 긍정 낮음
36 / 90 완료: 좌석·공간 긍정 낮음
37 / 90 완료: 분위기·소음 부정 보통
38 / 90 완료: 기타 부정 보통
39 / 90 완료: 청결 긍정 낮음
40 / 90 완료: 청결 긍정 낮음
41 / 90 완료: 좌석·공간 긍정 낮음
42 / 90 완료: 시설·비품 긍정 낮음

In [8]:
df["category"] = categories
df["ai_label"] = ai_labels
df["urgency"] = urgencies
df["reply"] = replies

df.to_csv("results.csv", index=False, encoding="utf-8-sig")

print(df["ai_label"].value_counts())
df.head()

ai_label
긍정        75
부정        14
감정: 긍정     1
Name: count, dtype: int64


,id,date,review,label,source,category,ai_label,urgency,reply
0,1,NaN,항상 잘 이용하고 있습니다. 앞으로도 자주 올게요,긍정,naver,기타,긍정,낮음,"안녕하세요, 카공족 건대로데오점을 항상 애용해 주셔서 정말 감사합니다! 앞으로도 편..."
1,2,NaN,화장실이 내부에 있어서 좋고 깔끔하게 관리가 잘 되고 있어요. 좋습니댱,긍정,naver,청결,긍정,낮음,"안녕하세요, 카공족 건대로데오점입니다! 매장 내부 화장실을 깔끔하게 이용하고 계시다..."
2,3,NaN,조용하고 깔끔해서 집중하기 좋아요,긍정,naver,좌석·공간,긍정,낮음,조용하고 깔끔한 분위기 속에서 집중이 잘 되셨다니 정말 다 기쁩니다! 😊 앞으로도 ...
3,4,NaN,매장이 깔끔하고 공부에 집중하기 너무 좋은 분위기였어요! 다음에 또 방문하고싶어요🤍,긍정,naver,청결,긍정,낮음,매장이 깔끔해서 공부하기에 좋은 분위기였다고 칭찬해 주셔서 정말 기쁩니다😊 앞으로도...
4,5,NaN,생각했던 것보다 훨씬 좋아요!!!! 조용하고 모두 열심히 집중하고 계시니 저도 덩달...,긍정,naver,분위기·소음,긍정,낮음,안녕하세요! 카공족 건대로데오점을 찾아주셔서 정말 감사합니다 😊 조용하고 집중하는 ...


In [9]:
print(df["category"].value_counts())
print(df["urgency"].value_counts())
df[df["ai_label"] == "감정: 긍정"]

category
좌석·공간           39
분위기·소음          17
청결              12
시설·비품           11
결제·요금            4
기타               3
카테고리: 분위기·소음     1
기분·소음            1
시설·공간            1
냉난방              1
Name: count, dtype: int64
urgency
낮음         76
보통          9
높음          4
긴급도: 낮음     1
Name: count, dtype: int64


,id,date,review,label,source,category,ai_label,urgency,reply
11,12,NaN,조용하고 깨끗한 분위가 조성되어있어 이용에 편리합니다.! 자격증준비하느라 집중할만한...,긍정,naver,카테고리: 분위기·소음,감정: 긍정,긴급도: 낮음,답글: 카공족 건대로데오점을 찾아주셔서 감사드립니다! 조용하고 깨끗한 분위기 속에서...


In [11]:
# ① 이름표 지우기 (12번 리뷰)
df["category"] = df["category"].str.replace("카테고리:", "").str.strip()
df["ai_label"] = df["ai_label"].str.replace("감정:", "").str.strip()
df["urgency"] = df["urgency"].str.replace("긴급도:", "").str.strip()
df["reply"] = df["reply"].str.replace("답글:", "").str.strip()

# ② 목록에 없는 카테고리 고치기
df["category"] = df["category"].str.replace("기분·소음", "분위기·소음")
df["category"] = df["category"].str.replace("시설·공간", "시설·비품")

# ③ 다시 저장하고 확인
df.to_csv("results.csv", index=False, encoding="utf-8-sig")
print(df["category"].value_counts())
print(df["ai_label"].value_counts())
print(df["urgency"].value_counts())

category
좌석·공간     39
분위기·소음    19
청결        12
시설·비품     12
결제·요금      4
기타         3
냉난방        1
Name: count, dtype: int64
ai_label
긍정    76
부정    14
Name: count, dtype: int64
urgency
낮음    77
보통     9
높음     4
Name: count, dtype: int64
